<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Colab notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href="https://youtu.be/nPMBfZWqPWI">intro video</a> to get started!

</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RapidFireAI/rapidfireai/blob/main/community_notebooks/sft-ecommerce-retail-chatbot.ipynb)

⚠️ **IMPORTANT:** Do not let the Colab notebook tab stay idle for more than 5 min; Colab will disconnect otherwise. Refresh the TensorBoard screen or interact with the cells to avoid disconnection.

# **🧠 Supervised Fine-Tuning (SFT) Experimentation for an E-Commerce Chatbot**

## What We're Building

We'll fine-tune an **e-commerce customer support chatbot** using two small language models (`gpt2` and `distilgpt2`) on the [Bitext Retail E-Commerce LLM Chatbot Training Dataset](https://huggingface.co/datasets/bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset). This public dataset contains instruction-response pairs covering common retail scenarios—refunds, shipping inquiries, order status, cancellations, and more.

Instead of training one configuration at a time, we use RapidFire AI to run **8 configurations concurrently** and compare their training curves side-by-side in TensorBoard—all on a single free-tier Colab GPU. The goal is to understand how **model choice, prompt formatting, and LoRA configuration** affect:

* Training stability and convergence behavior
* Instruction-following quality
* Evaluation metrics (BLEU, ROUGE-L)

## Our Approach

We use **Supervised Fine-Tuning (SFT)** with **LoRA (Low-Rank Adaptation)** to efficiently adapt pre-trained models for customer support. To find the best configuration, we systematically vary **three axes** across **8 total runs**:

- **2 base models**: `gpt2` (124M params) vs. `distilgpt2` (82M params) — both small enough for free-tier Colab
- **2 prompt formats**: Plain Q&A (`Question: ... Answer: ...`) vs. Instruction-style (`### Instruction: ... ### Response: ...`)
- **2 LoRA adapter sizes**: Small (rank 8, `c_attn` only) vs. Large (rank 32, `c_attn` + `c_proj`)

RapidFire AI's **chunk-based scheduling** trains all 8 configurations concurrently—splitting the dataset into chunks and letting every run train on each chunk before moving to the next. This provides comparative metrics early, instead of waiting for full sequential training to finish.

### What You'll Learn

- How to define and run multiple SFT experiments concurrently with RapidFire AI
- Using LoRA adapters of different capacities for parameter-efficient fine-tuning
- Comparing prompt formatting strategies for instruction-following tasks
- Monitoring all training curves simultaneously in TensorBoard
- Extracting and visualizing post-training metrics programmatically

## Install RapidFire AI Package and Setup

### Option 1: Install Locally (or on a VM)
For the full RapidFire AI experience—advanced experiment management, UI, and production features—we recommend installing the package on a machine you control (for example, a VM or your local machine) rather than Google Colab. See our [Install and Get Started](https://oss-docs.rapidfire.ai/en/latest/walkthrough.html) guide.

### Option 2: Install in Google Colab
For simplicity, you can run this notebook on Google Colab. The cell below installs `rapidfireai` and evaluation dependencies (`evaluate`, `sacrebleu`) if they are not already present, then runs `rapidfireai init` to set up the environment.

In [ ]:
import importlib.util, sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *pkgs])

if importlib.util.find_spec("rapidfireai") is None:
    pip_install(["rapidfireai"])
if importlib.util.find_spec("evaluate") is None:
    pip_install(["evaluate", "sacrebleu"])

!rapidfireai init

## Start RapidFire Services

RapidFire AI runs three backend services: the **Dispatcher** (REST API on port 8851), **MLflow** (experiment tracking on port 8852), and the **Frontend** dashboard (port 8853). The cell below checks whether these services are already running and launches them if not.

- If any issues arise, check the status in a terminal window using `rapidfireai status` or `rapidfireai doctor`.
- You can also run `rapidfireai start` from a terminal on Colab instead of the cell below.

In [ ]:
import socket
from time import sleep
import subprocess

def services_up():
    try:
        s = [socket.socket(socket.AF_INET, socket.SOCK_STREAM) for _ in range(3)]
        s[0].connect(("127.0.0.1", 8851))
        s[1].connect(("127.0.0.1", 8852))
        s[2].connect(("127.0.0.1", 8853))
        for x in s:
            x.close()
        return True
    except OSError:
        return False

if not services_up():
    subprocess.Popen(["rapidfireai", "start"])
    sleep(30)

print("RapidFire services running:", services_up())


## Reproducibility & Environment Setup

All random seeds are fixed to ensure **deterministic and reproducible experiments** across runs. This prevents variance from random initialization, data shuffling, or CUDA nondeterminism from affecting comparisons between configurations. We also disable tokenizer parallelism to avoid deadlocks in multiprocessing environments.

In [ ]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())


## Configure TensorBoard Backend

We use TensorBoard to visualize training loss, evaluation metrics, and text quality scores across all 8 runs simultaneously. Setting the `RF_TRACKING_BACKEND` environment variable tells RapidFire to log all metrics to TensorBoard, making every run visible, comparable, and persistent.

In [ ]:
os.environ["RF_TRACKING_BACKEND"] = "tensorboard"

## Import RapidFire Components

We import the core RapidFire classes:
- **`Experiment`**: Top-level container that groups runs, saves artifacts, and tracks metrics.
- **`RFGridSearch`**: Generates all combinations of your knobs into a Config Group.
- **`RFModelConfig`**: Wraps the base model, LoRA config, and training arguments into a single unit.
- **`RFLoraConfig` / `RFSFTConfig`**: RapidFire wrappers around PEFT's `LoraConfig` and TRL's `SFTConfig`, enabling grid-searchable parameters via `List()`.

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    RFGridSearch,
    RFModelConfig,
    RFLoraConfig,
    RFSFTConfig,
)
from datasets import load_dataset

## Load & Prepare Dataset

We use the [Bitext Retail E-Commerce LLM Chatbot Training Dataset](https://huggingface.co/datasets/bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset), which contains instruction-response pairs for common retail support scenarios (e.g., refund requests, shipping status, order cancellations).

The dataset is downsampled to **96 training** and **16 evaluation** examples to fit within free-tier Colab memory and runtime limits. For production training on a local machine, increase these ranges significantly (e.g., 10K+ samples).

In [ ]:
dataset = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset")

train_dataset = dataset["train"].select(range(96)).shuffle(seed=SEED)
eval_dataset  = dataset["train"].select(range(200, 216)).shuffle(seed=SEED)

## Define Prompt Formatting Variants

SFT training requires a consistent text format. We compare two prompt styles to study how formatting affects instruction-following behavior:

* **Plain Q&A** (`Question: ... Answer: ...`): Simple, direct format that mirrors natural question-answering.
* **Instruction-formatted** (`### Instruction: ... ### Response: ...`): Chat-style format with explicit section headers, commonly used in instruction-tuning pipelines.

Each example is formatted in both styles, and the formatting function is passed to `RFModelConfig` to be applied automatically during training.

In [ ]:
def add_prompts(ex):
    return {
        "text_qa":   f"Question: {ex['instruction']}\nAnswer: {ex['response']}",
        "text_inst": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}",
    }

train_dataset = train_dataset.map(add_prompts)
eval_dataset  = eval_dataset.map(add_prompts)

def format_qa(ex):   return {"text": ex["text_qa"]}
def format_inst(ex): return {"text": ex["text_inst"]}

## Define Evaluation Metrics

We define two text quality metrics to evaluate generated responses against reference answers:

- **ROUGE-L**: Measures the longest common subsequence between prediction and reference, capturing structural similarity and fluency.
- **BLEU**: Measures n-gram precision between prediction and reference, widely used for evaluating generated text quality.

Metrics are computed only when decoded text is available, ensuring robustness and avoiding invalid evaluations on raw logits.

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if not isinstance(preds, (list, tuple)) or not isinstance(labels, (list, tuple)):
        return {}
    if not preds or not labels:
        return {}
    if not isinstance(preds[0], str):
        return {}

    import evaluate
    rouge = evaluate.load("rouge")
    bleu  = evaluate.load("sacrebleu")

    r = rouge.compute(predictions=preds, references=labels, rouge_types=["rougeL"])
    b = bleu.compute(predictions=preds, references=[[x] for x in labels])

    return {
        "rougeL": float(r["rougeL"]),
        "bleu": float(b["score"]),
    }

## Initialize Experiment

Every RapidFire AI experiment needs a unique name. The `Experiment` object is the top-level container that groups all runs, saves artifacts, and tracks metrics. It automatically creates an MLflow experiment under the hood for structured logging. We use a timestamped name to avoid collisions across reruns.

In [ ]:
from datetime import datetime

experiment_name = f"sft-ecom-{datetime.now().strftime('%m%d-%H%M%S')}"
experiment = Experiment(experiment_name=experiment_name)

print("Experiment name:", experiment_name)


## Define LoRA (PEFT) Configurations

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique that adds small trainable matrices to frozen model layers instead of updating all weights. This drastically reduces memory usage and training time. Key parameters:

- **`r` (rank)**: Dimensionality of the low-rank adapter matrices. Higher values = more capacity but more memory. We test rank 8 (lightweight) vs. rank 32 (higher capacity).
- **`lora_alpha`**: Scaling factor for LoRA weights, typically set to 2x the rank.
- **`target_modules`**: Which model layers to adapt. We compare adapting only the combined attention projection (`c_attn`) vs. both attention and output projections (`c_attn` + `c_proj`).
- **`lora_dropout`**: Dropout rate applied to LoRA layers for regularization.

These two LoRA configs are wrapped in a `List()` so RapidFire AI will grid-search over them automatically.

In [ ]:
lora_knob = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn"],
        bias="none",
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"],
        bias="none",
    ),
])

## Define Model Configurations (8 Runs Total)

This is where RapidFire AI shines. Instead of hardcoding a single training configuration, we define a search space and let RapidFire run all combinations concurrently.

We create 4 `RFModelConfig` entries (2 models x 2 prompt formats), each with the `lora_knob` `List()` containing 2 LoRA configs. `RFGridSearch` will expand this into **4 x 2 = 8 total runs**:

| Run | Model | Prompt | LoRA |
|-----|-------|--------|------|
| 1 | gpt2 | Q&A | r=8, c_attn |
| 2 | gpt2 | Q&A | r=32, c_attn+c_proj |
| 3 | gpt2 | Instruction | r=8, c_attn |
| 4 | gpt2 | Instruction | r=32, c_attn+c_proj |
| 5 | distilgpt2 | Q&A | r=8, c_attn |
| 6 | distilgpt2 | Q&A | r=32, c_attn+c_proj |
| 7 | distilgpt2 | Instruction | r=8, c_attn |
| 8 | distilgpt2 | Instruction | r=32, c_attn+c_proj |

In [ ]:
def make_cfg(model, scheme, fmt):
    return RFModelConfig(
        model_name=model,
        peft_config=lora_knob,
        training_args=RFSFTConfig(
            learning_rate=3e-4,
            lr_scheduler_type="linear",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            max_steps=60,
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=5,
            per_device_eval_batch_size=2,
            fp16=True,
            gradient_checkpointing=True,
            report_to="tensorboard",
            run_name=f"{experiment_name}|{model}|{scheme}",
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False,
        },
        formatting_func=fmt,
        compute_metrics=compute_metrics,
    )

configs = List([
    make_cfg("gpt2",       "qa",   format_qa),
    make_cfg("gpt2",       "inst", format_inst),
    make_cfg("distilgpt2", "qa",   format_qa),
    make_cfg("distilgpt2", "inst", format_inst),
])


## Define Model Creation Function

RapidFire AI calls this function once per run to instantiate the model and tokenizer from the config dictionary. It receives the expanded config (with the specific LoRA/training args for that run) and must return a `(model, tokenizer)` tuple.

GPT-2 models don't have a dedicated pad token, so we set `pad_token = eos_token` and use left-padding, which is required for decoder-only causal LMs during batched generation.

In [ ]:
def create_model_fn(cfg):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model = AutoModelForCausalLM.from_pretrained(cfg["model_name"], **cfg["model_kwargs"])
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model.config.pad_token_id = model.config.eos_token_id

    return model, tokenizer


## Run Multi-Config Training (SFT)

Now we launch `run_fit()`—the main function for running multi-config training and evaluation. RapidFire AI will:

1. **Expand configs** into 8 individual runs (one per model x prompt x LoRA combination).
2. **Split the dataset** into `num_chunks=4` chunks for interleaved execution.
3. **Schedule all runs** on each chunk before moving to the next, so you get comparative metrics after the very first chunk.
4. **Log metrics** to TensorBoard in real time.

`num_chunks` controls the swap granularity: more chunks = more frequent comparisons but slightly more overhead from model swapping. 4–8 chunks works well for most experiments.

In [ ]:
experiment.run_fit(
    RFGridSearch(configs, trainer_type="SFT"),
    create_model_fn,
    train_dataset,
    eval_dataset,
    num_chunks=4,
    seed=SEED,
)

print("Training complete.")


## Launch TensorBoard

TensorBoard is launched only after event files exist, preventing empty dashboards. This notebook runs 8 SFT configurations sequentially, so metrics appear gradually:

- **Wait 3–5 minutes** after launching TensorBoard before proceeding to the next cells.
- **Refresh TensorBoard** using the refresh button within the TensorBoard portal itself.
- Seeing metrics for only a few runs at first is **normal and expected**—all 8 runs will appear once their training begins and logs are written.
- Do not assume missing runs mean failure.

In [ ]:
from pathlib import Path
import time

EXP_DIR = Path("/content/rapidfireai/rapidfire_experiments") / experiment_name
TB_DIR  = EXP_DIR / "tensorboard_logs"

print("Waiting for TensorBoard event files...")
for _ in range(60):
    if list(TB_DIR.rglob("events.out.tfevents*")):
        break
    time.sleep(2)

assert list(TB_DIR.rglob("events.out.tfevents*")), "No TensorBoard logs found!"

!pkill -f tensorboard || true

%load_ext tensorboard
%tensorboard --logdir {str(TB_DIR)} --port 6006

print("Done. TensorBoard is ready.")

## Extract Metrics from TensorBoard

After training, metrics are stored as TensorBoard event files (one directory per run). While TensorBoard is ideal for visual inspection, we also extract the **final logged value** of every scalar metric into a **pandas DataFrame** for tabular comparison, CSV export, and reporting.

Each RapidFire experiment writes logs under `/content/rapidfireai/rapidfire_experiments/<experiment_name>/tensorboard_logs/`. We dynamically resolve this path to ensure the notebook works across reruns.

In [ ]:
import pandas as pd
from pathlib import Path
from tensorboard.backend.event_processing import event_accumulator

TB_ROOT = Path("/content/rapidfireai/rapidfire_experiments")

TB_LOG_DIR = TB_ROOT / experiment_name / "tensorboard_logs"

print(f" Extracting metrics from: {TB_LOG_DIR}")

## Build Metrics Summary Table

The cell below iterates over all run directories, loads TensorBoard event files using TensorBoard's native API, automatically discovers all logged scalar metrics, and extracts the final value of each. The result is a single pandas DataFrame with one row per run and one column per metric—no metric names are hardcoded.

**Note:** Re-run this cell if you see incomplete results. Metrics for later runs take time to appear.

In [ ]:
import pandas as pd
from pathlib import Path
from tensorboard.backend.event_processing import event_accumulator

# 1. Dynamically find the logs for the experiment you just ran
TB_LOG_DIR = TB_ROOT / experiment_name / "tensorboard_logs"

all_results = []

print(f" Extracting metrics from: {TB_LOG_DIR}")

# Sorting by name ensures Run 1, 2, 3 order in the table
for run_dir in sorted(TB_LOG_DIR.iterdir(), key=lambda x: int(x.name) if x.name.isdigit() else x.name):
    if not run_dir.is_dir():
        continue

    ea = event_accumulator.EventAccumulator(str(run_dir), size_guidance={'scalars': 0})
    ea.Reload()

    tags = ea.Tags().get('scalars', [])
    if not tags:
        continue

    run_data = {"run": run_dir.name}

    for tag in tags:
        col_name = tag.replace('/', '_')

        values = ea.Scalars(tag)
        if values:
            run_data[col_name] = values[-1].value

    all_results.append(run_data)

if all_results:
    df = pd.DataFrame(all_results)

    cols = ['run'] + [c for c in df.columns if c != 'run']
    df = df[cols]

    print("\n METRICS SUMMARY TABLE (Including BLEU/ROUGE)")
    display(df)

    csv_path = f"{experiment_name}_results.csv"
    df.to_csv(csv_path, index=False)
    print(f" Saved summary to: {csv_path}")
else:
    print("\n No metrics found. Check if the training steps were enough to trigger logging.")

## Advanced Visual Analysis

This cell generates three presentation-ready artifacts from the metrics table:

1. **Multi-Metric Radar Chart** — Each run plotted as a normalized (0–1) profile across loss, BLEU, ROUGE-L, and accuracy. Exported as interactive HTML.
2. **Metric Correlation Heatmap** — Pairwise correlations between all logged metrics, highlighting redundancies and optimization dynamics. Saved as PNG.
3. **Styled Results Table** — Color-coded DataFrame emphasizing minimization (loss) and maximization (BLEU/ROUGE) for quick visual comparison.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

radar_metrics = [c for c in df.columns if any(m in c.lower() for m in ['loss', 'bleu', 'rouge', 'accuracy'])]
df_norm = df[radar_metrics].apply(lambda x: (x - x.min()) / (x.max() - x.min()))
df_norm['run'] = df['run']

fig_radar = go.Figure()
for _, row in df_norm.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=[row[m] for m in radar_metrics],
        theta=radar_metrics,
        fill='toself',
        name=f"Run {row['run']}"
    ))

fig_radar.update_layout(
    title="Run Profiles: Multi-Metric Comparison",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    template="plotly_dark"
)

plt.figure(figsize=(10, 8))
sns.heatmap(df.drop(columns=['run']).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Metric Correlation Heatmap (Feature Importance Insight)")
plt.savefig("metric_correlation.png")

styled_df = df.style.background_gradient(cmap='viridis', subset=[c for c in df.columns if 'loss' in c]) \
                    .background_gradient(cmap='YlGn', subset=[c for c in df.columns if any(m in c for m in ['bleu', 'rouge', 'accuracy'])]) \
                    .set_caption("Final Experiment Results: Ranked by Metric Performance")

print("GENERATING ADVANCED COMPETITION ARTIFACTS...")
fig_radar.show()
plt.show()
display(styled_df)

fig_radar.write_html(f"{experiment_name}_interactive_radar.html")
print(f" Saved Interactive Radar: {experiment_name}_interactive_radar.html")

# AUXILIARY CONTENT — METRICS, LOGS, DATASETS


In [ ]:
import shutil
from pathlib import Path
import zipfile
import os

# ----------- Locate experiment root -----------
RF_ROOT = Path("/content/rapidfireai")
EXP_ROOT = RF_ROOT / "rapidfire_experiments"

assert EXP_ROOT.exists(), "RapidFire experiments directory not found."

experiment_dirs = sorted(
    [p for p in EXP_ROOT.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)
assert experiment_dirs, "No experiments found."

EXP_DIR = experiment_dirs[0]
print(f" Using experiment: {EXP_DIR.name}")

BUNDLE_DIR = Path("/content/sft_submission_artifacts")
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)

# 1) TensorBoard Metrics (ALL RUNS)
TB_DIR = EXP_DIR / "tensorboard_logs"
assert TB_DIR.exists(), "TensorBoard logs not found."

tb_out = BUNDLE_DIR / "tensorboard_logs"
shutil.copytree(TB_DIR, tb_out)
print(" Copied TensorBoard event files (all runs)")

# 2) RapidFire + Training Logs
LOG_ROOT = RF_ROOT / "logs"

log_out = BUNDLE_DIR / "logs"
log_out.mkdir()

for log_name in ["rapidfire.log", "training.log"]:
    matches = list(LOG_ROOT.rglob(log_name))
    for i, log_file in enumerate(matches):
        dst = log_out / f"{log_file.parent.name}_{log_name}"
        shutil.copy(log_file, dst)

print(" Collected rapidfire.log and training.log files")


DATA_CACHE = Path("/root/.cache/huggingface/datasets")
data_out = BUNDLE_DIR / "dataset_cache"

if DATA_CACHE.exists():
    shutil.copytree(DATA_CACHE, data_out, dirs_exist_ok=True)
    print(" Copied HuggingFace dataset cache (metadata + shards)")
else:
    print(" No local dataset cache found (this is OK)")

#ZIP EVERYTHING
ZIP_PATH = Path("/content/SFT_Submission_Artifacts.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for file in BUNDLE_DIR.rglob("*"):
        z.write(file, arcname=file.relative_to(BUNDLE_DIR))

print("\n SUBMISSION ARTIFACTS READY")
print(f"ZIP file: {ZIP_PATH}")
print("\nContents include:")
print("- TensorBoard metrics (all runs)")
print("- rapidfire.log and training.log")
print("- Dataset cache / metadata (if available)")
print("\nUpload this ZIP to GitHub or share directly with judges.")


## Generate Metric Comparison Plots

Bar charts comparing eval loss, training loss, and text quality metrics (BLEU/ROUGE) across all 8 configurations. Saved as PNG screenshots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Output directory for screenshots
OUT_DIR = Path("/content/screenshots")
OUT_DIR.mkdir(exist_ok=True)

sns.set(style="whitegrid")


plt.figure(figsize=(10, 6))
sns.barplot(
    data=df,
    x="run",
    y="eval_loss",
    palette="tab10"
)
plt.title("Eval Loss Across All SFT Configurations")
plt.xlabel("Run")
plt.ylabel("Eval Loss")
plt.tight_layout()
plt.savefig(OUT_DIR / "eval_loss_all_configs.png", dpi=200)
plt.close()


plt.figure(figsize=(10, 6))
sns.barplot(
    data=df,
    x="run",
    y="loss",
    palette="tab10"
)
plt.title("Training Loss Across All SFT Configurations")
plt.xlabel("Run")
plt.ylabel("Training Loss")
plt.tight_layout()
plt.savefig(OUT_DIR / "training_loss_all_configs.png", dpi=200)
plt.close()


quality_cols = [c for c in df.columns if "bleu" in c.lower() or "rouge" in c.lower()]

if quality_cols:
    df_melt = df.melt(
        id_vars=["run"],
        value_vars=quality_cols,
        var_name="metric",
        value_name="score"
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=df_melt,
        x="run",
        y="score",
        hue="metric",
        palette="Set2"
    )
    plt.title("Text Quality Metrics Across All SFT Configurations")
    plt.xlabel("Run")
    plt.ylabel("Score")
    plt.legend(title="Metric")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "bleu_rouge_all_configs.png", dpi=200)
    plt.close()


print(" Screenshot-ready metric plots saved:")
for f in OUT_DIR.iterdir():
    print(" -", f.name)

## Conclusion

In this notebook, we fine-tuned an e-commerce customer support chatbot by running **8 SFT configurations concurrently** on a single free-tier Colab GPU using RapidFire AI. Here's what we covered:

1. **Defined a Config Group** using `List()` wrappers and `RFGridSearch` to specify multiple models, prompt formats, and LoRA adapters.
2. **Ran all configurations concurrently** with `run_fit()`, getting comparative metrics after each data chunk instead of waiting for full training.
3. **Monitored training in real time** via TensorBoard, comparing loss curves and eval metrics side-by-side.
4. **Extracted and visualized metrics** programmatically with radar charts, correlation heatmaps, and styled tables.

### Key Findings

Across all runs, the strongest configuration was **distilgpt2 + Instruction-style prompts + LoRA r=32** (c_attn + c_proj), which achieved the lowest eval loss and best convergence stability. Higher LoRA rank consistently improved instruction-following, and instruction-style formatting outperformed plain Q&A on generation metrics.

### Next Steps

- **Try different models**: Replace gpt2/distilgpt2 with larger models like [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), [Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B), or [Phi-3-mini](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct).
- **Expand the grid**: Add more learning rates, LoRA ranks, or target module combinations to explore a wider hyperparameter space.
- **Scale up the dataset**: Increase from 96 training samples to 10K+ for production-quality fine-tuning on a local machine or VM.
- **Explore other training methods**: Use `RFDPOConfig` or `RFGRPOConfig` for preference alignment (DPO/GRPO) instead of SFT.
- **Use the RapidFire UI**: For a richer monitoring experience beyond TensorBoard, run RapidFire locally with the full dashboard at `http://localhost:8853`.

For more details, see the [RapidFire AI documentation](https://oss-docs.rapidfire.ai).